In [ ]:
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_agentchat.messages import TextMessage, MultiModalMessage
from autogen_core import Image as AGImage

from PIL import Image
from io import BytesIO
import requests
from dotenv import load_dotenv
import asyncio
import os

load_dotenv()

True

### defining the model_client

In [2]:
model_client = OpenAIChatCompletionClient(model='gpt-4o')

# TextMessage

#### defining the agent by giving the name and system message to act

In [3]:
agent = AssistantAgent(name = 'text_agent',
                       model_client=model_client,
                       system_message='you are helpful agent'
                       )

In [4]:
async def test_text_message():
    text_message = TextMessage(content='what is the capital of usa',source='user')
    result = await agent.run(task=text_message)
    print(result.messages[-1].content)
await test_text_message()    

The capital of the United States is Washington, D.C.


# multimodal messages

In [5]:
async def test_multi_modal_messages():
    response = requests.get('https://picsum.photos/id/15/200/300')
    pil_image = Image.open(BytesIO(response.content))
    ag_image = AGImage(pil_image)

    multi_modal_message = MultiModalMessage(
        content=['what is in the image', ag_image],
        source='user'
    )
    result = await agent.run(task=multi_modal_message)
    print(result.messages[-1].content)

await test_multi_modal_messages()

The image shows a waterfall surrounded by lush greenery and rocky terrain. There are rocks and pebbles in the foreground leading up to the waterfall in the background.


# Structured output

In [6]:
from pydantic import BaseModel


In [7]:
class PlanetInfo(BaseModel):
    name : str
    color : str
    distance_miles : int


In [8]:
## structured model_client
model_client = OpenAIChatCompletionClient(
    model='gpt-4o',
    response_format = PlanetInfo
)

## Unstructured model_client

unstructured_model_client = OpenAIChatCompletionClient(
    model='gpt-4o'
)

# no need to define the api_keys since it is loaded from the load_dotenv()

# using unsturtured model output

In [9]:
agent  = AssistantAgent(
    name='planet_agent',
    model_client=unstructured_model_client,
    system_message='you are helpful agent and gives response in json format' \
    '{name: str}' \
    'age: int' \
    '}',

)

In [ ]:
async def test_unstructured_output():
    task = TextMessage(content="please give info about planet mars",source='user')
    result = await agent.run(task=task)
    structured_response = result.messages[-1].content
    print(structured_response)

await test_unstructured_output()

Mars is the fourth planet from the Sun in our solar system. It is known for its reddish appearance, which is due to iron oxide, commonly known as rust, on its surface. Mars is often referred to as the "Red Planet." Here are some key features of Mars:

- **Diameter**: Approximately 6,779 kilometers (4,212 miles), making it about half the size of Earth.
- **Atmosphere**: Composed mostly of carbon dioxide, with nitrogen and argon. The atmosphere is thin, with a surface pressure less than 1% of Earth's.
- **Moons**: Mars has two small moons, Phobos and Deimos.
- **Surface**: Features include the largest volcano in the solar system, Olympus Mons, and the longest canyon, Valles Marineris.
- **Temperature**: Can vary widely from -125 degrees Celsius (-195 degrees Fahrenheit) during winter at the poles, to 20 degrees Celsius (68 degrees Fahrenheit) during summer at the equator.
- **Exploration**: Numerous missions have been sent to explore Mars by orbiters, landers, and rovers, including NASA'

# Structured model client using 

In [12]:
agent  = AssistantAgent(
    name='planet_agent',
    model_client= model_client,
    system_message='you are helpful agent and gives response in json format')


async def test_structured_output():
    task = TextMessage(content="please give info about planet mars",source='user')
    result = await agent.run(task=task)
    structured_response = result.messages[-1].content
    print(structured_response)

await test_structured_output()

{"name":"Mars","color":"Red","distance_miles":141600000}


/Users/sivavenu/Desktop/AI_projects/autogen_projects/.venv/lib/python3.12/site-packages/pydantic/main.py:464: UserWarning: Pydantic serializer warnings:
  PydanticSerializationUnexpectedValue(Expected `none` - serialized value may not be as expected [field_name='parsed', input_value=PlanetInfo(name='Mars', c...istance_miles=141600000), input_type=PlanetInfo])
  return self.__pydantic_serializer__.to_python(
